# Lab 6: Mouse Electrophysiology: An Introduction to LFPs

#### Stephen Spivack, CDS Adjunct, Fall 2025

In this lab we:

1. Compute mean and SEM plots for LFP traces  
   - By session and stimulus condition  
   - By group  
2. Apply digital signal processing to individual trials
   - Used for smoothing of time-series traces
   - Gaussian filter
3. Curve fitting of individual trials
   - Used for modeling, dimensionality reduction, etc.
   - Cubic splines


Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import LSQUnivariateSpline, BSpline

Init client

In [ ]:
%load_ext google.cloud.bigquery
client = bigquery.Client() 

Ingest data

In [ ]:
%%bigquery auditory_cortex
    select
      session,
      condition,
      frequency,
      amplitude,
      trial_num,
      array(select safe_cast(x as float64) from unnest(split(trace, ',')) as x) as trace
    from neural-ds-fe73.lab6_mouse_lfp.auditory_cortex

Compute summary statistic plots for each session, stimulus condition

In [ ]:
# --- Extract unique sessions for looping ---
sessions = sorted(auditory_cortex["session"].unique())
peak_response_times = []
avg_trace = []
all_traces = []
group_condition = []

# --- Loop over sessions ---
for session in sessions:
    try:
        session_df = auditory_cortex[auditory_cortex["session"] == session].copy()
        condition = session_df["condition"].iloc[0]
        freqs = sorted(session_df["frequency"].unique())
        amps = sorted(session_df["amplitude"].unique())

        n_freq = len(freqs)
        n_amp = len(amps)

        # --- Initialize per-session figure ---
        plt.figure(figsize=(10, 15))
        counter = 1
        t = np.linspace(0, 5, 5001)
        peak_magnitude = []
        peak_time = []
        temp_avg_trace = []
        freq_counter = []
        amp_counter = []
        temp_trials = []

        # --- Nested loop over frequency x amplitude grid ---
        for i, freq in enumerate(freqs):
            for j, amp in enumerate(amps):
                subset = session_df[
                    (session_df["frequency"] == freq) &
                    (session_df["amplitude"] == amp)
                ]
                if subset.empty:
                    continue

                # Stack trial traces into [time x trials]
                trials = np.stack(subset["trace"].values, axis=1)
                baseline = trials[:2000, :].mean(axis=0)
                normalized = trials - baseline
                avg = np.nanmean(normalized, axis=1)
                sem = np.nanstd(normalized, axis=1) / np.sqrt(normalized.shape[1])

                peak_magnitude.append(abs(np.min(avg[2000:4000])))
                peak_time.append(np.argmin(normalized[2000:4000, :], axis=0))
                temp_avg_trace.append(avg)
                freq_counter.append(freq)
                amp_counter.append(amp)
                temp_trials.append(normalized)

                # --- Subplot per frequency × amplitude ---
                plt.subplot(n_freq, n_amp, counter)
                color = "dodgerblue" if condition == "WT" else "orange"
                plt.plot(t, avg, color=color)
                plt.fill_between(t, avg + sem, avg - sem, color="lightgrey")
                plt.plot([2, 2], [np.min(avg - sem), np.max(avg + sem)],
                         color="limegreen", lw=5)
                plt.title(f"freq={freq}, amp={amp}")
                counter += 1

        plt.tight_layout()
        plt.show()

        # --- Report per-session summary ---
        print(f"\nSession: {session} | Condition: {condition}")
        print(f"Optimal frequency: {freq_counter[np.argmax(peak_magnitude)]}")
        print(f"Optimal amplitude: {amp_counter[np.argmax(peak_magnitude)]}")

        opt_peak_time = peak_time[np.argmax(peak_magnitude)]
        opt_peak_time = np.delete(opt_peak_time, np.argwhere(opt_peak_time <= 0))
        peak_response_times.append(opt_peak_time)
        avg_trace.append(np.array(temp_avg_trace)[np.argmax(peak_magnitude)])
        all_traces.append(temp_trials[np.argmax(peak_magnitude)])
        group_condition.append(condition)

    except Exception as e:
        print(f"[Session {session}] Error: {e}")


Compute average and SEM for fmr1ko versus wt

In [ ]:
# --- Compute FMR1 average and SEM ---
fmr1_mask = np.argwhere(np.array(group_condition) == 'FMR1').flatten()
fmr1_avg = np.array(avg_trace)[fmr1_mask].mean(axis=0)
fmr1_sem = np.array(avg_trace)[fmr1_mask].std(axis=0) / np.sqrt(len(fmr1_mask))

# --- Compute WT average and SEM ---
wt_mask = np.argwhere(np.array(group_condition) == 'WT').flatten()
wt_avg = np.array(avg_trace)[wt_mask].mean(axis=0)
wt_sem = np.array(avg_trace)[wt_mask].std(axis=0) / np.sqrt(len(wt_mask))

# --- Plot ---
t = np.linspace(0, 5, 5001)
plt.figure(figsize=(14, 7))

# --- FMR1KO plot ---
plt.subplot(1, 2, 1)
plt.plot(t, fmr1_avg, color='orange', label='fmr1ko')
plt.fill_between(t, fmr1_avg + fmr1_sem, fmr1_avg - fmr1_sem, color='lightgrey', label='sem')
plt.plot([2, 2], [-8, 2], color='limegreen', label='tone', lw=5)
plt.ylim([-9, 2.5])
plt.xlabel('Time (s)')
plt.ylabel('Change in voltage (mV)')
plt.title('FMR1 Knockout (n = {})'.format(len(fmr1_mask)))
plt.legend()

# --- WT plot ---
plt.subplot(1, 2, 2)
plt.plot(t, wt_avg, color='dodgerblue', label='control')
plt.fill_between(t, wt_avg + wt_sem, wt_avg - wt_sem, color='lightgrey', label='sem')
plt.plot([2, 2], [-8, 2], color='limegreen', label='tone', lw=5)
plt.ylim([-9, 2.5])
plt.xlabel('Time (s)')
plt.ylabel('Change in voltage (mV)')
plt.title('Wild Type Control (n = {})'.format(len(wt_mask)))
plt.legend()

plt.tight_layout()
plt.show()


## Gaussian Smoothing of LFP Traces

To reduce high-frequency noise while preserving biologically meaningful dynamics, we apply **Gaussian smoothing** to each LFP trace. This involves convolving the raw voltage signal $V(t)$ with a Gaussian kernel $G_\sigma(t)$:

$$
G_\sigma(t) = \frac{1}{\sqrt{2\pi}\sigma} \, e^{-\frac{t^2}{2\sigma^2}}, \quad 
\tilde{V}(t) = (V * G_\sigma)(t)
$$

**Where:**
- $V(t)$: raw voltage trace  
- $\tilde{V}(t)$: smoothed trace  
- $\sigma$: standard deviation of the kernel, in **samples** (not seconds)

---

### Choosing the Right $\sigma$

The value of $\sigma$ determines the temporal resolution of the smoothing:

- Small $\sigma$ (e.g., 5–10) — preserves fast transients and spike-like features  
- Medium $\sigma$ (20–50) — balances smoothing and temporal fidelity  
- Large $\sigma$ (100+) — emphasizes slow fluctuations and suppresses sharp events  

---

### Implementation
```python
from scipy.ndimage import gaussian_filter1d

# Apply Gaussian smoothing
smoothed = gaussian_filter1d(trace, sigma=50)  # 50-sample (~50 ms) smoothing.


In [ ]:
# Sample WT and FMR1 trials (random)
wt_df = auditory_cortex[auditory_cortex["condition"] == "WT"].sample(n=18, random_state=0)
fmr1_df = auditory_cortex[auditory_cortex["condition"] == "FMR1"].sample(n=18, random_state=0)

# Combine
combined_df = pd.concat([wt_df, fmr1_df], ignore_index=True).reset_index(drop=True)

# Prepare plotting variables
t = np.linspace(0, 5, 5001)
sigma = 50  # Gaussian smoothing factor
num_plots = len(wt_df) + len(fmr1_df)

# Create grid plot
fig, axes = plt.subplots(6, 6, figsize=(22, 22), sharex=True, sharey=True)
for i, ax in enumerate(axes.flat):
    row = combined_df.iloc[i]
    condition = row["condition"]
    
    trace = np.array(row["trace"], dtype=float)
    smoothed = gaussian_filter1d(trace, sigma=sigma)

    color = "dodgerblue" if condition == "WT" else "orange"
    ax.plot(t, smoothed, color=color, lw=1)
    ax.axvline(2, color="limegreen", lw=1, linestyle='--')  # Tone onset
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{condition} | Trial {row['trial_num']}", fontsize=8)

fig.suptitle(f"Smoothed LFP Traces (Top: WT, Bottom: FMR1KO) — Gaussian σ={sigma}", fontsize=20)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()


## Spline Fitting of LFP Traces

To estimate the smooth underlying structure of each LFP trace, we fit a **regression B-spline** to the signal within a 3-second window (2–5 s).  
We use a cubic spline with a single interior **knot** placed at the trial's minimum (peak response latency).

---

### Univariate B-Spline Regression

We approximate the voltage trace $V(t)$ over the interval $t \in [0, 3000]$ samples using a **cubic spline** of the form:

$$
\tilde{V}(t) = \sum_{i=0}^{n} c_i \cdot B_i^{(k)}(t)
$$

Where:
- $\tilde{V}(t)$ is the spline-approximated voltage trace  
- $B_i^{(k)}(t)$ is the $i$‑th basis function of order $k$ (here, $k=3$ for cubic)  
- $c_i$ are the spline coefficients (fit to each trial)  
- The knot vector is padded to ensure boundary conditions for cubic B-splines
- $n=5$ basis functions because cubic splines with 1 internal knot and 4 repeated boundary knots at each end results in 5 active spline segments

---

### Knot Placement

To allow the spline to flexibly capture the evoked response:

- We place a **single internal knot** at the time of minimum voltage (trough):
  
$$
\text{knot} = \arg\min_t V(t), \quad t \in [0, 3000]
$$

- This allows the spline to "bend" at the peak response, improving fit quality

---

### Implementation
```python
import numpy as np
from scipy.interpolate import LSQUnivariateSpline, BSpline

# Example data
x = np.linspace(0, 3000, 3000)   # time in samples
y = np.sin(x / 300) + np.random.normal(0, 0.1, len(x))  # mock LFP trace

# Fit cubic spline with a single internal knot at the trough
knot = [np.argmin(y)]
spl = LSQUnivariateSpline(x, y, knot)

# Convert LSQ spline to explicit BSpline form
t_knot = spl.get_knots()[1]
coeffs = spl.get_coeffs()
t_padded = [0]*4 + [t_knot] + [3000]*4
bs = BSpline(t_padded, coeffs, 3)

# Evaluate fitted spline
y_fit = bs(x)


In [ ]:
# Sample WT and FMR1 trials
wt_df = auditory_cortex[auditory_cortex["condition"] == "WT"].sample(n=18, random_state=1)
fmr1_df = auditory_cortex[auditory_cortex["condition"] == "FMR1"].sample(n=18, random_state=1)
combined_df = pd.concat([wt_df, fmr1_df], ignore_index=True).reset_index(drop=True)

# Prepare plotting variables
t = np.linspace(0, 5, 5001)
x = np.linspace(0, 2999, 3000)  # spline window

# Create grid plot
fig, axes = plt.subplots(6, 6, figsize=(24, 24), sharex=True, sharey=True)
for i, ax in enumerate(axes.flat):
    row = combined_df.iloc[i]
    raw = np.array(row["trace"], dtype=float)
    y = raw[2000:5000]

    try:
        # Fit spline with single trough knot
        knot = [np.argmin(y)]
        spl = LSQUnivariateSpline(x, y, knot)
        t_knot = spl.get_knots()[1]
        c = spl.get_coeffs()
        t_padded = [0]*4 + [t_knot] + [3000]*4
        bs = BSpline(t_padded, c, 3)
        y_fit = bs(x)
    except:
        y_fit = None

    # Plot raw trace
    if row["condition"] == "WT":
        ax.plot(t, raw, color='dodgerblue', lw=0.6, alpha=0.8)
    else:
        ax.plot(t, raw, color='orange', lw=0.6, alpha=0.8)

    # Plot spline fit
    if y_fit is not None:
        ax.plot(t[2000:5000], y_fit, color='red', lw=1.5)

    ax.axvline(2, color='limegreen', lw=1, linestyle='--')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{row['condition']} | Trial {row['trial_num']}", fontsize=8)

fig.suptitle("Spline Fits over Raw Traces (WT = Blue, FMR1 = Orange, Fit = Red)", fontsize=22)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()